# آموزش مدل حذف (A) و دسته‌بندی (B) — کالاهای دیجیتال
داده از برنچ `arena/01a081fa-web-scrapper` ریپوی `Mooli-web/web-scrapper` می‌آید.
نوت‌بوک خودکفاست: ریپو را کلون می‌کند، داده را export می‌کند، دو مدل را آموزش و ارزیابی می‌کند.
**Runtime → GPU لازم نیست** (پایه CPU است)؛ اگر خواستی ترنسفورمر، سلول آخر را اجرا کن.

In [ ]:
# 1) نصب و کلون و export
!pip -q install scikit-learn joblib
!git -q clone https://github.com/Mooli-web/web-scrapper -b arena/01a081fa-web-scrapper repo
%cd repo/5_unified_local_hub
!python ml_stats.py --export
print("آماده.")

In [ ]:
# 2) کتابخانه‌ها + نرمال‌سازی + بارگذاری
import json, re, unicodedata, numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix

_DIG=str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩","01234567890123456789")
_LET=str.maketrans({"ي":"ی","ى":"ی","ك":"ک","ک":"ک","ة":"ه","أ":"ا","إ":"ا","آ":"ا"})
_INV={ord(c):None for c in "\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069"}
def norm(t):
    t=(t or "").translate(_INV)
    t="".join(c for c in t if not(unicodedata.category(c)=="Mn" or c=="\u0640"))
    t=t.replace("\u200c"," ").translate(_DIG).translate(_LET)
    return re.sub(r"\s+"," ",re.sub(r"[^\w\s]+"," ",t)).strip().lower()

REP=re.compile(r"^(\d)\1{5,}$")
def load(f): return [json.loads(l) for l in open(f,encoding="utf-8") if l.strip()]
def nums(rows):
    return np.array([[np.log1p(r.get("price") or 0),1.0 if (r.get("price") or 0)==0 else 0.0,
                      1.0 if REP.match(str(r.get("price") or 0)) else 0.0,len(r["title"])] for r in rows])
A=load("exports/ml/quality_train.jsonl"); B=load("exports/ml/category_train.jsonl")
canon={json.loads(l)["id"]:json.loads(l).get("canonical_key") or "" for l in open("exports/training_bundle/listings.jsonl",encoding="utf-8")}
print("نمونه A:",len(A),"| نمونه B:",len(B))

In [ ]:
# 3) تقسیم بر canonical_key (بدون نشت) + ویژگی‌سازی
def feats(rows):
    X=[norm(r["title"]) for r in rows]
    c=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,max_features=60000).fit_transform(X)
    w=TfidfVectorizer(analyzer="word",ngram_range=(1,2),min_df=2,max_features=30000).fit_transform(X)
    return hstack([c,w,csr_matrix(nums(rows))]),(c,w)
def split(rows):
    k2i=defaultdict(list)
    for i,r in enumerate(rows): k2i[canon.get(r["id"],r["id"])].append(i)
    ks=list(k2i); tr,te=train_test_split(ks,test_size=0.2,random_state=42)
    tr=[i for k in tr for i in k2i[k]]; te=[i for k in te for i in k2i[k]]
    return tr,te

In [ ]:
# 4) تسک A — حذف/نگه‌داشتن
XA,_=feats(A); yA=np.array([r["label"] for r in A]); tr,te=split(A)
mA=LogisticRegression(class_weight="balanced",max_iter=2000).fit(XA[tr],yA[tr])
pA=mA.predict(XA[te])
print(f"A  ورودی={XA.shape[1]:,} بعد  خروجی=2")
print(f"   accuracy={accuracy_score(yA[te],pA):.3f}  macroF1={f1_score(yA[te],pA,average='macro'):.3f}")

In [ ]:
# 5) تسک B — دسته‌بندی ۱۶ کلاس
cats=sorted({r["label"] for r in B}); c2i={c:i for i,c in enumerate(cats)}
XB,_=feats(B); yB=np.array([c2i[r["label"]] for r in B]); trb,teb=split(B)
mB=LogisticRegression(class_weight="balanced",max_iter=3000).fit(XB[trb],yB[trb])
pB=mB.predict(XB[teb])
print(f"B  ورودی={XB.shape[1]:,} بعد  خروجی={len(cats)}")
print(f"   accuracy={accuracy_score(yB[teb],pB):.3f}  macroF1={f1_score(yB[teb],pB,average='macro'):.3f}")
print(classification_report(yB[teb],pB,target_names=cats,digits=2))

In [ ]:
# 6) ذخیره‌ی مدل‌ها برای دانلود
import joblib, google.colab.files as files
joblib.dump({"model":mA,"classes":list(mA.classes_)},"model_quality.pkl")
joblib.dump({"model":mB,"classes":cats},"model_category.pkl")
files.download("model_quality.pkl"); files.download("model_category.pkl")
print("مدل‌ها ذخیره و دانلود شدند.")

## تفسیر و قدم بعد
- **ورودی:** عنوان نرمال‌شده → TF-IDF (char ۳-۵ + word ۱-۲) + ۴ ویژگی عددی. **خروجی:** A=۲، B=۱.
- **معیار:** macro-F1 و per-class (نه accuracy، چون کلاس‌ها نابرابرند).
- **انتظار:** A ~.95 / B ~.94 macro-F1. ضعیف‌ترین‌ها `other` و `accessories` (کم‌نمونه).
- **اگر بالاتر خواستی:** ترنسفورمر (xlm-roberta / bert-fa) روی GPU؛ همان داده، سرِ ۲ یا ۱ کلاسه.